<a href="https://colab.research.google.com/github/anjalidrj/GO-analysis-of-peptides/blob/main/common_peptides_pathways.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Common peptides across crops — 6 pathways
Photosynthesis, stress, nitrogen, carbon fixation, chlorophyll, gibberellin.
Run cells 1-8 in order. Add/remove a pathway by editing `PATHWAY_ROOTS` in Cell 1 only.

In [ ]:
# ==================== CELL 1 — setup + upload ====================
!pip -q install goatools >/dev/null 2>&1

import os, re, glob, gzip, urllib.request
import pandas as pd
from collections import defaultdict

# --- all pathways to target: name -> GO root (edit here only) ---
PATHWAY_ROOTS = {
    "photosynthesis": "GO:0015979",   # photosynthesis
    "stress":         "GO:0006950",   # response to stress
    "nitrogen":       "GO:0042128",   # nitrate assimilation
    "carbon":         "GO:0019253",   # Calvin cycle / carbon fixation
    "chlorophyll":    "GO:0015995",   # chlorophyll biosynthesis
    "ga":             "GO:0009686",   # gibberellin biosynthesis (GO:0009739 = GA response)
}
MIN_CROPS    = 3              # "common" = pathway hit in >= this many crops
COTTON_TAXID = "3635"         # Gossypium hirsutum (native gene2go)

from google.colab import files
print("Upload all 9 files: 5 exact_hits_final_*.csv + 4 gProfiler_*_athaliana*.csv")
files.upload()

def find(pattern):
    hits = glob.glob(pattern)
    return hits[0] if hits else None

CROP_FILES = {
    "soy":    find("exact_hits_final_soyB.csv"),
    "chilli": find("exact_hits_final_caB.csv"),
    "cotton": find("exact_hits_final_cottonB.csv"),
    "grape":  find("exact_hits_final_grapeB.csv"),
    "tomato": find("exact_hits_final_solB.csv"),
}
ORTH_FILES = {
    "soy":    find("gProfiler_gmax_athaliana_B.csv"),
    "chilli": find("gProfiler_cannuum_athaliana_B.csv"),
    "grape":  find("gProfiler_vvinifera_athaliana_B.csv"),
    "tomato": find("gProfiler_slycopersicum_athaliana_B.csv"),
}
print("\nCrop hit tables found:")
for k, v in CROP_FILES.items(): print(f"  {k:7s}: {v}")
print("Ortholog maps found:")
for k, v in ORTH_FILES.items(): print(f"  {k:7s}: {v}")


Upload all 9 files: 5 exact_hits_final_*.csv + 4 gProfiler_*_athaliana*.csv


Saving exact_hits_final_caB.csv to exact_hits_final_caB.csv
Saving exact_hits_final_cottonB.csv to exact_hits_final_cottonB.csv
Saving exact_hits_final_grapeB.csv to exact_hits_final_grapeB.csv
Saving exact_hits_final_solB.csv to exact_hits_final_solB.csv
Saving exact_hits_final_soyB.csv to exact_hits_final_soyB.csv
Saving gProfiler_cannuum_athaliana_B.csv to gProfiler_cannuum_athaliana_B.csv
Saving gProfiler_gmax_athaliana_B.csv to gProfiler_gmax_athaliana_B.csv
Saving gProfiler_slycopersicum_athaliana_B.csv to gProfiler_slycopersicum_athaliana_B.csv
Saving gProfiler_vvinifera_athaliana_B.csv to gProfiler_vvinifera_athaliana_B.csv

Crop hit tables found:
  soy    : exact_hits_final_soyB.csv
  chilli : exact_hits_final_caB.csv
  cotton : exact_hits_final_cottonB.csv
  grape  : exact_hits_final_grapeB.csv
  tomato : exact_hits_final_solB.csv
Ortholog maps found:
  soy    : gProfiler_gmax_athaliana_B.csv
  chilli : gProfiler_cannuum_athaliana_B.csv
  grape  : gProfiler_vvinifera_athalian

In [ ]:
# ==================== CELL 2 — GO ontology + pathway subtrees ====================
from goatools.obo_parser import GODag

if not os.path.exists("go-basic.obo") or os.path.getsize("go-basic.obo") < 10_000_000:
    !wget -c -O go-basic.obo "https://release.geneontology.org/2024-01-17/ontology/go-basic.obo"

# load WITH relationships so part_of links (needed for photosynthesis) are included
godag = GODag("go-basic.obo", optional_attrs={"relationship"})

def subtree(root):
    """All descendants of root via is_a + part_of, walking downward."""
    seen, stack = set(), [root]
    while stack:
        cur = stack.pop()
        if cur in seen:
            continue
        seen.add(cur)
        node = godag.get(cur)
        if not node:
            continue
        for ch in node.children:
            stack.append(ch.id)
        for rel_set in getattr(node, "relationship_rev", {}).values():
            for ch in rel_set:
                stack.append(ch.id)
    return seen

# build a subtree for every pathway
pathway_terms = {name: subtree(root) for name, root in PATHWAY_ROOTS.items()}
for name, terms in pathway_terms.items():
    print(f"{name:15s}: {len(terms):5d} terms")


--2026-07-09 04:41:57--  https://release.geneontology.org/2024-01-17/ontology/go-basic.obo
Resolving release.geneontology.org (release.geneontology.org)... 172.64.145.49, 104.18.42.207, 2606:4700:4400::ac40:9131, ...
Connecting to release.geneontology.org (release.geneontology.org)|172.64.145.49|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 31183660 (30M) [text/obo]
Saving to: ‘go-basic.obo’

go-basic.obo        100%[===================>]  29.74M  39.9MB/s    in 0.7s    

2026-07-09 04:41:58 (39.9 MB/s) - ‘go-basic.obo’ saved [31183660/31183660]

go-basic.obo: fmt(1.2) rel(2024-01-17) 45,869 Terms; optional_attrs(relationship)
photosynthesis :    36 terms
stress         :  1280 terms
nitrogen       :     2 terms
carbon         :     3 terms
chlorophyll    :    12 terms
ga             :     5 terms


In [ ]:
# ==================== CELL 3 — Arabidopsis AT-locus -> GO ====================
ARAB_GAF = "tair.gaf.gz"
if not os.path.exists(ARAB_GAF):
    !wget -c -O tair.gaf.gz "https://current.geneontology.org/annotations/tair.gaf.gz"

at_go = defaultdict(set)
with gzip.open(ARAB_GAF, "rt") as fh:
    for line in fh:
        if line.startswith("!"):
            continue
        c = line.rstrip("\n").split("\t")
        if len(c) < 11:
            continue
        go = c[4]
        for token in [c[1], c[2], *c[10].split("|")]:
            if re.match(r"AT[1-5MC]G\d{5}", token, re.I):
                at_go[token.upper()].add(go)

print(f"Arabidopsis AT loci with GO: {len(at_go)}")
ex = next(iter(at_go)); print("example:", ex, "->", list(at_go[ex])[:3])


--2026-07-09 04:42:09--  https://current.geneontology.org/annotations/tair.gaf.gz
Resolving current.geneontology.org (current.geneontology.org)... 104.18.42.207, 172.64.145.49, 2606:4700:4400::ac40:9131, ...
Connecting to current.geneontology.org (current.geneontology.org)|104.18.42.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7218381 (6.9M) [application/gzip]
Saving to: ‘tair.gaf.gz’

tair.gaf.gz         100%[===================>]   6.88M  --.-KB/s    in 0.08s   

2026-07-09 04:42:09 (83.5 MB/s) - ‘tair.gaf.gz’ saved [7218381/7218381]

Arabidopsis AT loci with GO: 22220
example: AT1G01010 -> ['GO:0003677', 'GO:0005634', 'GO:0016020']


In [ ]:
# ==================== CELL 4 — cotton LOC -> GO (native NCBI) ====================
if not os.path.exists("gene2go.gz"):
    print("Downloading NCBI gene2go (~1.3 GB, several min)...")
    !wget -c -O gene2go.gz "https://ftp.ncbi.nlm.nih.gov/gene/DATA/gene2go.gz"

cotton_go = defaultdict(set)
with gzip.open("gene2go.gz", "rt") as fh:
    for line in fh:
        if line.startswith("#"):
            continue
        p = line.rstrip("\n").split("\t")
        if p[0] != COTTON_TAXID:
            continue
        cotton_go["LOC" + p[1]].add(p[2])

print(f"Cotton LOC genes with GO: {len(cotton_go)}")
ex = next(iter(cotton_go)); print("example:", ex, "->", list(cotton_go[ex])[:3])


--2026-07-09 04:42:17--  https://ftp.ncbi.nlm.nih.gov/gene/DATA/gene2go.gz
Resolving ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)... 130.14.250.10, 130.14.250.11, 2607:f220:41e:250::10, ...
Connecting to ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)|130.14.250.10|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1363221883 (1.3G) [application/x-gzip]
Saving to: ‘gene2go.gz’

gene2go.gz          100%[===================>]   1.27G  61.8MB/s    in 13s     

2026-07-09 04:42:30 (98.2 MB/s) - ‘gene2go.gz’ saved [1363221883/1363221883]

Cotton LOC genes with GO: 50002
example: LOC107885926 -> ['GO:0051287', 'GO:0006739', 'GO:0000287']


In [ ]:
# ==================== CELL 5 — load ortholog maps: crop gene -> {AT} ====================
def load_orth(path):
    m = defaultdict(set)
    if not path or not os.path.exists(path):
        return m
    df = pd.read_csv(path)
    for a, at in zip(df["initial_alias"], df["ortholog_ensg"]):
        at = str(at)
        if at.startswith("AT"):
            m[str(a).upper()].add(at.upper())
    return m

orth = {c: load_orth(p) for c, p in ORTH_FILES.items()}
for c, m in orth.items():
    print(f"{c:7s}: {len(m)} genes with an AT ortholog")


soy    : 10184 genes with an AT ortholog
chilli : 3805 genes with an AT ortholog
grape  : 3766 genes with an AT ortholog
tomato : 2385 genes with an AT ortholog


In [ ]:
# ==================== CELL 6 — per-crop gene_id extractor ====================
def clean_gene(crop, row):
    gd  = str(row.get("gene_description", ""))
    gid = str(row.get("gene_id", ""))
    if crop == "cotton":
        m = re.search(r"\((LOC\d+)\)", gd)
        return m.group(1) if m else None
    if crop == "tomato":
        m = re.search(r"gene:gene-(\S+)", gd)
        return (m.group(1) if m else gid.replace("gene-", "")).upper()
    if crop == "grape":
        m = re.search(r"gene:(\S+)", gd)
        g = (m.group(1) if m else gid).upper()
        return g.replace("VITIS", "VITVI", 1)   # match the Vitvi ortholog keys
    return gid.upper()   # soy (GLYMA_..), chilli (T459_..)

for crop, path in CROP_FILES.items():
    if not path:
        print(f"{crop:7s}: NO FILE"); continue
    r0 = pd.read_csv(path, nrows=1).iloc[0]
    print(f"{crop:7s}: {clean_gene(crop, r0)}")


soy    : GLYMA_19G117400
chilli : T459_22635
cotton : LOC107922178
grape  : VITVI05G01235
tomato : SOLYC09G091610.3


In [ ]:
# ==================== CELL 7 — tag peptides by pathway, per crop ====================
results = {name: defaultdict(lambda: defaultdict(set)) for name in PATHWAY_ROOTS}

def go_for_gene(crop, gene):
    if crop == "cotton":
        return cotton_go.get(gene, set())
    s = set()
    for at in orth[crop].get(gene, ()):
        s |= at_go.get(at, set())
    return s

for crop, path in CROP_FILES.items():
    if not path:
        continue
    df = pd.read_csv(path)
    for _, row in df.iterrows():
        gene = clean_gene(crop, row)
        if not gene:
            continue
        gs = go_for_gene(crop, gene)
        if not gs:
            continue
        pep = str(row["peptide_id"])
        for name, terms in pathway_terms.items():
            if gs & terms:
                results[name][crop][pep].add(gene)
    print(f"{crop:7s}: " + " | ".join(
        f"{name[:4]} {len(results[name][crop]):4d}" for name in PATHWAY_ROOTS))


soy    : phot   99 | stre 1633 | nitr    6 | carb   11 | chlo   38 | ga   33
chilli : phot   59 | stre  854 | nitr    7 | carb    2 | chlo   19 | ga   13
cotton : phot   35 | stre  559 | nitr    0 | carb    3 | chlo    7 | ga    8
grape  : phot   79 | stre  852 | nitr    2 | carb    8 | chlo   22 | ga    8
tomato : phot   31 | stre  570 | nitr    8 | carb    1 | chlo   11 | ga   11


In [ ]:
# ==================== CELL 8 — intersect across crops + save (with seq) ====================
pep_seq = {}
for crop, path in CROP_FILES.items():
    if not path:
        continue
    d = pd.read_csv(path, usecols=lambda c: c in ("peptide_id", "peptide_seq"))
    for pid, seq in zip(d["peptide_id"].astype(str), d["peptide_seq"].astype(str)):
        pep_seq.setdefault(pid, seq)

MIN_CROPS_OUT = MIN_CROPS

def summarize(pathway):
    per = results[pathway]
    crops = [c for c in CROP_FILES if per.get(c)]
    pep_crops, pep_genes = defaultdict(set), defaultdict(dict)
    for crop in crops:
        for pep, genes in per[crop].items():
            pep_crops[pep].add(crop)
            pep_genes[pep][crop] = ";".join(sorted(genes))
    rows = []
    for pep, cset in pep_crops.items():
        if len(cset) >= MIN_CROPS_OUT:
            row = {"peptide_id": pep, "peptide_seq": pep_seq.get(pep, ""),
                   "n_crops": len(cset)}
            for crop in CROP_FILES:
                row[crop] = pep_genes[pep].get(crop, "")
            rows.append(row)
    out = pd.DataFrame(rows)
    if out.empty:
        return pd.DataFrame(columns=["peptide_id", "peptide_seq", "n_crops"] + list(CROP_FILES))
    return out.sort_values("n_crops", ascending=False)

def report(name, out):
    print(f"\n=== {name}: {len(out)} peptides in >= {MIN_CROPS_OUT} crops ===")
    if out.empty:
        print("  (none)"); return
    for n in (5, 4, 3):
        print(f"  {n} crops: {(out['n_crops'] == n).sum()}")
    print(out.head(15).to_string(index=False))

outputs = {}
for name in PATHWAY_ROOTS:
    out = summarize(name)
    fname = f"common_peptides_{name}.csv"
    out.to_csv(fname, index=False)
    outputs[name] = out
    report(name.upper(), out)

for name in PATHWAY_ROOTS:
    files.download(f"common_peptides_{name}.csv")



=== PHOTOSYNTHESIS: 9 peptides in >= 3 crops ===
  5 crops: 0
  4 crops: 0
  3 crops: 9
           peptide_id peptide_seq  n_crops                                                                                             soy                chilli cotton grape           tomato
 ATQ70527.1_pep57_mc0      PNARIM        3                                                                                 GLYMA_11G119300            T459_25473              SOLYC10G051310.2
ATQ70428.1_pep279_mc0     RGRSGRQ        3                                                                 GLYMA_02G007700;GLYMA_10G137800            T459_02451              SOLYC01G080840.3
ATQ70428.1_pep274_mc0     SRRIDNQ        3                                                                 GLYMA_02G007700;GLYMA_10G137800            T459_02451              SOLYC01G080840.3
ATQ69651.1_pep194_mc0      GGRVAE        3                                                                 GLYMA_04G019100;GLYMA_06G019400         

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>